In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
file_path = "Maindata.xlsx" 
df = pd.read_excel(file_path)

In [4]:
df.head()

,Review_ID,Review_Text,Rating,Source,Sentiment
0,1,Very satisfied with the performance,5,Flipkart,Positive
1,2,Product is okay nothing special,3,Amazon,Neutral
2,3,Terrible experience overall 😤,2,Instagram,Negative
3,4,Terrible experience overall,2,Voice,Negative
4,5,Not worth the price,2,Amazon,Negative


In [5]:
df.isnull().sum()

Review_ID      0
Review_Text    0
Rating         0
Source         0
Sentiment      0
dtype: int64

In [6]:
df = df[["Review_Text", "Sentiment"]]

In [7]:
df.sample(5)

,Review_Text,Sentiment
10228,Honestly It works fine but could be better 🙂,Neutral
3097,Average performance 🙂,Neutral
3617,Worth the money,Positive
10474,Okay so Very satisfied with the performance 😍,Positive
11240,Average performance 🙂,Neutral


In [8]:
label_mapping = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}

df["label"] = df["Sentiment"].map(label_mapping)

df["label"].value_counts()

label
0    4091
2    3955
1    3954
Name: count, dtype: int64

In [9]:
X = df["Review_Text"]
y = df["label"]

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, stratify=y, random_state=42
)

In [11]:
X_train = X_train.tolist()
X_test = X_test.tolist()
y_train = y_train.tolist()
y_test = y_test.tolist()

In [12]:
X_train

['Honestly Terrible experience overall 👎',
 'Honestly Absolutely loved the experience',
 'Decent for the price',
 'Very disappointed with the product 😤',
 'Honestly Very satisfied with the performance',
 'So basically Very satisfied with the performance 💯',
 'Terrible experience overall 💔',
 'Absolutely loved the experience 😊',
 'It works fine but could be better',
 'This product is really good and works as expected',
 'This product is really good and works as expected',
 'Quality is very poor',
 'It stopped working after few days',
 'Very satisfied with the performance 😊',
 'Umm Worth the money',
 'Worth the money',
 'Yeah actually Not bad not great',
 'It works fine but could be better',
 'So basically Average performance 😐',
 'So basically Very disappointed with the product 😭',
 'Honestly Not bad not great',
 'Highly recommended for everyone 🔥',
 'Product is okay nothing special 🤔',
 'Absolutely loved the experience',
 'Terrible experience overall 😭',
 'Product is okay nothing speci

In [13]:
# Load pretrained BERT tokenizer.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased") ## Converts texts into proper numbers and token ids to be understood by pretrained BERT

C:\Users\chauh\anaconda3\envs\dl_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [15]:
train_dataset = SentimentDataset(X_train, y_train, tokenizer)
test_dataset = SentimentDataset(X_test, y_test, tokenizer)

In [16]:
trainloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
testloader = DataLoader(test_dataset, batch_size=16)

In [17]:
batch = next(iter(trainloader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

print("Input IDs shape:", input_ids.shape)
print("Attention mask shape:", attention_mask.shape)
print("Labels shape:", labels.shape)
print("Device:", input_ids.device)

Input IDs shape: torch.Size([16, 128])
Attention mask shape: torch.Size([16, 128])
Labels shape: torch.Size([16])
Device: cuda:0


In [30]:
##Load Model (Pretrained BERT)

from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

model.to(device)

# Freeze BERT layers
for param in model.bert.parameters():
    param.requires_grad = False

Loading weights: 100%|████████████████| 199/199 [00:01<00:00, 160.20it/s, Materializing param=bert.pooler.dense.weight]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initializ

In [37]:
optimizer = AdamW(model.parameters(), lr=2e-4) ## Critetia or CrossEntropyLoss is calculated automatically

In [38]:
from tqdm import tqdm

In [39]:
epochs = 14

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch in tqdm(trainloader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    avg_loss = total_loss / len(trainloader)

    print(f"\nEpoch {epoch+1}/{epochs}")
    print("Training Loss:", avg_loss)

100%|████████████████████████████████████████████████████████████████████████████████| 600/600 [00:59<00:00, 10.15it/s]



Epoch 1/14
Training Loss: 0.8956637543439865


100%|████████████████████████████████████████████████████████████████████████████████| 600/600 [00:51<00:00, 11.62it/s]



Epoch 2/14
Training Loss: 0.8037018783887228


100%|████████████████████████████████████████████████████████████████████████████████| 600/600 [00:56<00:00, 10.57it/s]


...


Epoch 12/14
Training Loss: 0.4388032826781273


100%|████████████████████████████████████████████████████████████████████████████████| 600/600 [01:07<00:00,  8.93it/s]



Epoch 13/14
Training Loss: 0.42390022886296114


100%|████████████████████████████████████████████████████████████████████████████████| 600/600 [00:49<00:00, 12.03it/s]


Epoch 14/14
Training Loss: 0.41296167343854906


In [40]:
from sklearn.metrics import accuracy_score

model.eval()

preds = []
labels_all = []

with torch.no_grad():
    for batch in testloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask)

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1)

        preds.extend(predictions.cpu().numpy())
        labels_all.extend(labels.cpu().numpy())

accuracy = accuracy_score(labels_all, preds)
print("Validation Accuracy:", accuracy)

Validation Accuracy: 0.9870833333333333


In [42]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(labels_all, preds))
print(confusion_matrix(labels_all, preds))

              precision    recall  f1-score   support

           0       1.00      0.98      0.99       818
           1       0.96      1.00      0.98       791
           2       1.00      0.98      0.99       791

    accuracy                           0.99      2400
   macro avg       0.99      0.99      0.99      2400
weighted avg       0.99      0.99      0.99      2400

[[801  17   0]
 [  0 791   0]
 [  2  12 777]]


In [43]:
torch.save(model.state_dict(), "bert_sentiment_model.pth")